# Inspect generated RAW products — data-store explorer

Browse everything the pipeline wrote into the data-store: canonical L0 products (compressed ISPs),
open-container L0s, L1A′ decodes, the calibration campaign (`caldb/`), quicklooks and phase reports.

- Store root: `$S2_DATA_STORE` (default `~/data-store`) — same convention as `scripts/run_pipeline.py`.
- Needs only `numpy + zarr + matplotlib` (the plain generator env; no eopf).
- Regenerate products first if the store is empty: `python scripts/run_pipeline.py` (nominal → `l0/`)
  or `python scripts/run_pipeline.py calibration` (campaign → `caldb/`).

In [ ]:
# One-time kernel setup — safe to re-run. After installing, RESTART the kernel.
import sys

_need_pkg = _need_deps = False
try:
    import s2_msi_raw_generator  # noqa: F401
except Exception:
    _need_pkg = True
try:
    import matplotlib.pyplot  # noqa: F401
    import zarr  # noqa: F401
except Exception:
    _need_deps = True
if _need_pkg:
    !{sys.executable} -m pip install --quiet -e .. --no-deps
if _need_deps:
    !{sys.executable} -m pip install --quiet matplotlib zarr
    !{sys.executable} -m pip install --quiet --force-reinstall --no-deps matplotlib zarr
if _need_pkg or _need_deps:
    print("installed/repaired — RESTART the kernel (Kernel ▸ Restart Kernel…) and run all cells again")
else:
    print("environment OK")

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import zarr

from s2_msi_raw_generator import l0product, sensor

STORE = Path(os.environ.get("S2_DATA_STORE") or "~/data-store").expanduser()


def du_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6 if p.exists() else 0.0


print(f"store: {STORE}")
for sub in ("inputs", "caldb", "l0", "l1a_prime", "l1b", "quicklook", "figures", "report"):
    d = STORE / sub
    n = sum(1 for _ in d.iterdir()) if d.is_dir() else 0
    print(f"  {sub:10s} {n:4d} entries  {du_mb(d):9.1f} MB")

## Product inventory

All PSFD-named `.zarr` products per store layer. Nominal L0s (canonical + `_OC` open container)
live under `l0/`; the calibration campaign L0s (`S02MSIDCA`/`S02MSISCA`) + cal-DB ADFs under `caldb/`.

In [ ]:
products = {}
for sub in ("l0", "caldb", "l1a_prime", "l1b", "inputs"):
    entries = sorted((STORE / sub).glob("*.zarr")) if (STORE / sub).is_dir() else []
    products[sub] = entries
    print(f"— {sub}/ ({len(entries)} products)")
    for p in entries:
        print(f"    {du_mb(p):9.1f} MB  {p.name}")

canonical = [p for p in products["l0"] if "_OC" not in p.name] + [
    p for p in products["caldb"] if p.name.startswith("S02MSI")
]
open_containers = [p for p in products["l0"] if "_OC" in p.name]
print(f"\ncanonical L0s: {len(canonical)}, open containers: {len(open_containers)}, "
      f"L1A': {len(products['l1a_prime'])}")

## Product tree

Zarr hierarchy of a product (works with zarr-python 2 and 3). Point `tree()` at any product path.

In [ ]:
def tree(group, prefix="", depth=0, max_depth=4):
    """Version-agnostic zarr tree print (arrays with shape/dtype, then subgroups)."""
    for name, arr in sorted(group.arrays(), key=lambda kv: kv[0]):
        print(f"{prefix}{name}  {arr.shape} {arr.dtype}")
    if depth + 1 > max_depth:
        return
    for name, sub in sorted(group.groups(), key=lambda kv: kv[0]):
        print(f"{prefix}{name}/")
        tree(sub, prefix + "    ", depth + 1, max_depth)


if canonical:
    prod = canonical[0]
    print(prod.name, "\n")
    tree(zarr.open_group(str(prod), mode="r"))
else:
    print("no canonical L0 in the store yet — run the pipeline first")

## Root metadata + achieved compression

STAC discovery properties, the acquisition configuration and the **real** per-band CCSDS-122
compression ratios stamped by the packaging phase (REQ-FUNC-092).

In [ ]:
if canonical:
    g = zarr.open_group(str(canonical[0]), mode="r")
    attrs = dict(g.attrs)
    props = attrs["stac_discovery"]["properties"]
    keep = ("datetime", "start_datetime", "end_datetime", "platform", "eopf:type",
            "product:type", "msi:datatake_type", "sat:relative_orbit", "eopf:datastrip_id")
    for k in keep:
        if k in props:
            print(f"{k:22s} {props[k]}")
    acq = attrs["other_metadata"]["sensor_configuration"]["acquisition_configuration"]
    print(f"{'operation_mode':22s} {acq.get('operation_mode')}")
    print(f"{'compression_scheme':22s} {acq.get('compression_scheme', '—')}")

    print("\nper-band compression (from measurements/*/*/attrs):")
    print(f"  {'band':6s} {'ratio':>6s} {'raw MB':>8s} {'comp MB':>8s} {'packets':>8s}")
    for dname, det in sorted(g["measurements"].groups(), key=lambda kv: kv[0]):
        for bkey, bg in sorted(det.groups(), key=lambda kv: kv[0]):
            a = dict(bg.attrs)
            if "compression" in a:
                c = a["compression"]
                print(f"  {dname}/{bkey:4s} {c['ratio']:6.2f} {c['raw_bytes']/1e6:8.1f} "
                      f"{c['compressed_bytes']/1e6:8.1f} {a.get('n_packets', 0):8d}")

## Band images (stored DN)

Products written with `store_decoded=True` (the default) carry the decoded `band{NN}` arrays —
instant to display. Percentile-stretched (2–98 %) so texture is visible.

In [ ]:
def stored_bands(product):
    """Yield (det, band_key, array) for every stored decoded band of a product."""
    g = zarr.open_group(str(product), mode="r")
    if "measurements" not in g:
        return
    for dname, det in sorted(g["measurements"].groups(), key=lambda kv: kv[0]):
        for bkey, bg in sorted(det.groups(), key=lambda kv: kv[0]):
            for aname, arr in sorted(bg.arrays(), key=lambda kv: kv[0]):
                if aname.startswith("band"):
                    yield dname, bkey, arr


def show(img, title, max_lines=2048):
    img = np.asarray(img[:max_lines])
    lo, hi = np.percentile(img, (2, 98))
    plt.figure(figsize=(9, 5))
    plt.imshow(np.clip(img, lo, hi), cmap="gray", aspect="auto", interpolation="nearest")
    plt.title(f"{title}  [{lo:.0f}–{hi:.0f} DN]")
    plt.xlabel("column"); plt.ylabel("line"); plt.colorbar(label="DN")
    plt.tight_layout(); plt.show()


shown = 0
for prod in canonical:
    for dname, bkey, arr in stored_bands(prod):
        show(arr, f"{prod.name} — {dname}/{bkey}")
        shown += 1
        break  # first band per product; drop this break to walk all bands
    if shown >= 3:
        break
if not shown:
    print("no stored decoded bands (products may be ISP-only) — use the ISP decode cell below")

## Ground-decode from the ISP stream (verification)

Decode one band from the **compressed CCSDS space packets** and check it equals the stored DN —
the same bit-exact contract `run_pipeline.py`'s ground-decode phase asserts.
*The codec is pure Python: a full-height band takes a while; small/cal products are instant.*

In [ ]:
if canonical:
    prod = canonical[-1]  # cal products (smallest) come last in the combined list
    g = zarr.open_group(str(prod), mode="r")
    dname, det_g = next(iter(sorted(g["measurements"].groups(), key=lambda kv: kv[0])))
    bkey = next(iter(sorted(dict(det_g.groups()))))
    det = int(dname[1:])
    bname = next(b for b in sensor.BANDS if sensor.zarr_band_key(b) == bkey)
    dn = l0product.read_l0_isp_dn(str(prod), det, bname)
    print(f"{prod.name} {dname}/{bkey}: decoded {dn.shape} {dn.dtype}")
    stored = next((a for d2, b2, a in stored_bands(prod) if (d2, b2) == (dname, bkey)), None)
    if stored is not None:
        print("bit-exact vs stored DN:", bool(np.array_equal(dn, np.asarray(stored))))
    show(dn, f"ISP-decoded — {prod.name} {dname}/{bkey}")

## Calibration campaign + cal-DB

The dark (`S02MSIDCA`, CSM closed) and sun-diffuser (`S02MSISCA`, Lambertian) acquisitions,
and the Option-Y NUC gains derived from those very frames.

In [ ]:
for prefix, label in (("S02MSIDCA", "dark (DASC)"), ("S02MSISCA", "sun-diffuser (ABSR)")):
    hits = sorted((STORE / "caldb").glob(f"{prefix}*.zarr")) or sorted((STORE / "l0").glob(f"{prefix}*.zarr"))
    if hits:
        for dname, bkey, arr in stored_bands(hits[0]):
            show(arr, f"{label} — {hits[0].name} {dname}/{bkey}", max_lines=512)
            break
    else:
        print(f"no {prefix} product (run: python scripts/run_pipeline.py calibration)")

nuc_p = STORE / "caldb" / "nuc.zarr"
if nuc_p.exists():
    nuc = zarr.open_group(str(nuc_p), mode="r")
    plt.figure(figsize=(9, 4))
    for bn, arr in sorted(nuc["gain"].arrays(), key=lambda kv: kv[0]):
        plt.plot(np.asarray(arr), label=bn, lw=1)
    plt.title("cal-DB NUC gain per detector column"); plt.xlabel("column"); plt.ylabel("gain")
    plt.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()
else:
    print("no caldb/nuc.zarr yet")

## Quicklooks

PNGs written by the `quicklook` phase (L1A RGB strips, DWT subband showcase, cal frames).

In [ ]:
from IPython.display import Image, display

pngs = sorted((STORE / "quicklook").glob("*.png")) if (STORE / "quicklook").is_dir() else []
print(f"{len(pngs)} quicklooks")
for p in pngs[:8]:
    print(p.name)
    display(Image(filename=str(p), width=700))

## Phase reports

Every phase drops its JSON under `report/`; `e2e_report.md` + `summary.json` aggregate them.

In [ ]:
rep = STORE / "report"
for p in (sorted(rep.glob("*.json")) if rep.is_dir() else []):
    print(f"— {p.name}")

gd = rep / "ground_decode.json"
if gd.exists():
    data = json.loads(gd.read_text())
    print(f"\nground-decode: {'band':6s} {'bit_exact':>9s} {'ratio':>6s} {'packets':>8s}  decoder")
    for bn, r in data.items():
        print(f"               {bn:6s} {str(r.get('bit_exact')):>9s} {r.get('ratio', 0):6.2f} "
              f"{r.get('n_packets', 0):8d}  {r.get('decoder', '?')}")

va = rep / "validate.json"
if va.exists():
    data = json.loads(va.read_text())
    print(f"\nvalidate:      {'band':6s} {'bit_identical':>13s} {'lines_lost':>10s}")
    for bn, r in data.items():
        print(f"               {bn:6s} {str(r.get('bit_identical_kept')):>13s} {r.get('lines_lost', '?'):>10}")

md = rep / "e2e_report.md"
if md.exists():
    print("\n" + "=" * 60 + "\n" + md.read_text()[:2000])